### Cell1 - Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### Cell 2 — create the watermark table

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS mia_catalog.gold._pipeline_state (
        source_table STRING,
        target_table STRING,
        last_processed_version BIGINT,
        updated_at TIMESTAMP
    )
    USING DELTA
""")
print("mia_catalog.gold._pipeline_state ready")

mia_catalog.gold._pipeline_state ready


### DIM_PRODUCT TABLE CREATION

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS mia_catalog.gold.dim_product (
        product_sk INT,
        product_key STRING,
        product_name STRING,
        category STRING,
        price DOUBLE,
        brand STRING,
        stock_quantity INT,
        source_system STRING,
        record_hash STRING,
        effective_start_date DATE,
        effective_end_date DATE,
        is_current BOOLEAN,
        dw_created_at TIMESTAMP,
        dw_updated_at TIMESTAMP
    )
    USING DELTA
""")
print("mia_catalog.gold.dim_product ready (created if it didn't already exist)")

mia_catalog.gold.dim_product ready (created if it didn't already exist)


### Cell 3 — helper functions (get/set watermark, get current version)

In [0]:
def get_last_processed_version(source_table: str, target_table: str) -> int:
    result = spark.sql(f"""
        SELECT last_processed_version
        FROM mia_catalog.gold._pipeline_state
        WHERE source_table = '{source_table}' AND target_table = '{target_table}'
    """).collect()
    if len(result) == 0:
        return -1
    return result[0]["last_processed_version"]


def set_last_processed_version(source_table: str, target_table: str, version: int):
    spark.sql(f"""
        MERGE INTO mia_catalog.gold._pipeline_state AS target
        USING (SELECT '{source_table}' AS source_table, '{target_table}' AS target_table,
                      {version} AS last_processed_version, current_timestamp() AS updated_at) AS source
        ON target.source_table = source.source_table AND target.target_table = source.target_table
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


def get_current_table_version(table_name: str) -> int:
    return spark.sql(f"DESCRIBE HISTORY {table_name}").selectExpr("max(version)").collect()[0][0]

In [0]:
print(
    get_last_processed_version(
        "mia_catalog.silver.silver_customers",
        "mia_catalog.gold.dim_product"
    )
)

-1


In [0]:
spark.sql(f"DESCRIBE HISTORY mia_catalog.silver.silver_products").selectExpr("max(version)").collect()[0][0]

10

### Cell 4 — backfill the watermark (run once)

In [0]:
dim_product_row_count = spark.sql("SELECT count(*) as c FROM mia_catalog.gold.dim_product").collect()[0]["c"]

if dim_product_row_count == 0:
    print("dim_product is empty — running initial full load instead of incremental")

    silver_products_current = spark.table("mia_catalog.silver.silver_products")
    window_spec = Window.orderBy("product_key")

    dim_product_initial = (
        silver_products_current
        .select("product_key", "product_name", "category", "price", "brand",
                 "stock_quantity", "source_system", "record_hash")
        .withColumn("product_sk", row_number().over(window_spec))
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("dw_created_at", current_timestamp())
        .withColumn("dw_updated_at", current_timestamp())
        .select("product_sk", "product_key", "product_name", "category", "price",
                 "brand", "stock_quantity", "source_system", "record_hash",
                 "effective_start_date", "effective_end_date", "is_current",
                 "dw_created_at", "dw_updated_at")
    )

    dim_product_initial.write.format("delta").mode("append").saveAsTable("mia_catalog.gold.dim_product")
    print(f"Initial load complete: {dim_product_initial.count()} rows")
else:
    print(f"dim_product already has {dim_product_row_count} rows — skipping initial load")

# Backfill/confirm the watermark either way, so the incremental cells below know where to start
current_version_before_changes = get_current_table_version("mia_catalog.silver.silver_products")
set_last_processed_version("mia_catalog.silver.silver_products", "mia_catalog.gold.dim_product", current_version_before_changes)
print(f"Watermark set at silver_products version {current_version_before_changes}")

dim_product already has 1210 rows — skipping initial load
Watermark set at silver_products version 10


### Cell 5 — pick 15 existing products to simulate a price change on

In [0]:
sample_product_keys = [row["product_key"] for row in
    spark.sql("SELECT product_key FROM mia_catalog.silver.silver_products LIMIT 15").collect()]

print(f"Simulating price changes for {len(sample_product_keys)} existing products")
print(sample_product_keys)

Simulating price changes for 15 existing products
['e82143ae-f458-4251-aa44-f86bb534174e', 'cba03dd6-168f-4d77-b5d6-9087874dd445', 'ba306da1-7036-4289-b823-268a75ab8df6', '96f58f07-5f38-4bbc-8d37-6651c8a6a7b0', '468626f5-7df2-4f66-a180-c178b8681355', 'fe24e923-9bed-470d-9e9b-bbe31726bbc0', 'cc4df993-8e33-4912-8626-57fae4fc2e7f', 'f022c94b-fb92-42c4-b222-1c0d4fd6f222', '4fe459ba-c44b-4dd9-962f-26dcc404bed3', 'f2823689-0f6d-48be-8f08-82ea9442d3e4', '721403bd-54d6-444e-b2eb-c5423ed6b3b4', '79e8bcc9-b27d-4b56-b026-408c6414f66f', 'e7768bef-dbed-4961-b8df-0fb47bd86f2a', 'a6113c61-7489-4725-9ebc-5409fe640150', '02ffd9da-2a25-480a-9ddf-2acb113f462a']


### Cell 6 — apply the simulated update directly to Silver

In [0]:
keys_sql_list = ", ".join([f"'{k}'" for k in sample_product_keys])
print(keys_sql_list)

spark.sql(f"""
    UPDATE mia_catalog.silver.silver_products
    SET price = price * 1.15,
        record_hash = md5(concat_ws('|', product_name, category, cast(price * 1.15 as string),
                                     brand, cast(stock_quantity as string),
                                     cast(rating as string), cast(discount_percentage as string))),
        last_updated_ts = current_timestamp()
    WHERE product_key IN ({keys_sql_list})
""")

print("Simulated update applied")

'e82143ae-f458-4251-aa44-f86bb534174e', 'cba03dd6-168f-4d77-b5d6-9087874dd445', 'ba306da1-7036-4289-b823-268a75ab8df6', '96f58f07-5f38-4bbc-8d37-6651c8a6a7b0', '468626f5-7df2-4f66-a180-c178b8681355', 'fe24e923-9bed-470d-9e9b-bbe31726bbc0', 'cc4df993-8e33-4912-8626-57fae4fc2e7f', 'f022c94b-fb92-42c4-b222-1c0d4fd6f222', '4fe459ba-c44b-4dd9-962f-26dcc404bed3', 'f2823689-0f6d-48be-8f08-82ea9442d3e4', '721403bd-54d6-444e-b2eb-c5423ed6b3b4', '79e8bcc9-b27d-4b56-b026-408c6414f66f', 'e7768bef-dbed-4961-b8df-0fb47bd86f2a', 'a6113c61-7489-4725-9ebc-5409fe640150', '02ffd9da-2a25-480a-9ddf-2acb113f462a'
Simulated update applied


### Cell 7 — confirm CDF captured it

In [0]:
new_silver_version = get_current_table_version("mia_catalog.silver.silver_products")
print(f"silver_products is now at version {new_silver_version} (was {current_version_before_changes})")

display(spark.sql(f"""
    SELECT product_key, product_name, price, _change_type, _commit_version, _commit_timestamp
    FROM table_changes('mia_catalog.silver.silver_products', {current_version_before_changes + 1}, {new_silver_version})
    ORDER BY _commit_version
"""))

silver_products is now at version 11 (was 10)


product_key,product_name,price,_change_type,_commit_version,_commit_timestamp
f2823689-0f6d-48be-8f08-82ea9442d3e4,Lemon Garlic Shrimp,8.99,update_preimage,11,2026-08-15T11:53:48.000Z
fe24e923-9bed-470d-9e9b-bbe31726bbc0,Kids' Educational Tablet,149.4885,update_postimage,11,2026-08-15T11:53:48.000Z
cba03dd6-168f-4d77-b5d6-9087874dd445,Puzzle,32.1885,update_postimage,11,2026-08-15T11:53:48.000Z
96f58f07-5f38-4bbc-8d37-6651c8a6a7b0,Handmade Leather Journal,45.9885,update_postimage,11,2026-08-15T11:53:48.000Z
ba306da1-7036-4289-b823-268a75ab8df6,Electric Food Slicer,114.98849999999999,update_postimage,11,2026-08-15T11:53:48.000Z
cc4df993-8e33-4912-8626-57fae4fc2e7f,Peach Mango Smoothie,3.49,update_preimage,11,2026-08-15T11:53:48.000Z
f2823689-0f6d-48be-8f08-82ea9442d3e4,Lemon Garlic Shrimp,10.3385,update_postimage,11,2026-08-15T11:53:48.000Z
96f58f07-5f38-4bbc-8d37-6651c8a6a7b0,Handmade Leather Journal,39.99,update_preimage,11,2026-08-15T11:53:48.000Z
f022c94b-fb92-42c4-b222-1c0d4fd6f222,Water Bottle with Built-in Fruit Infuser,18.99,update_preimage,11,2026-08-15T11:53:48.000Z
468626f5-7df2-4f66-a180-c178b8681355,Portable Pet Water Bottle,18.99,update_preimage,11,2026-08-15T11:53:48.000Z


### Cell 8 — read the change window (insert/update_postimage only)

In [0]:
silver_changes = spark.sql(f"""
    SELECT product_key, product_name, category, price, brand, stock_quantity,
           source_system, record_hash
    FROM table_changes('mia_catalog.silver.silver_products', {current_version_before_changes + 1}, {new_silver_version})
    WHERE _change_type IN ('insert', 'update_postimage')
""")

print(f"Changed/new rows to process into Gold: {silver_changes.count()}")
display(silver_changes)

Changed/new rows to process into Gold: 15


product_key,product_name,category,price,brand,stock_quantity,source_system,record_hash
e82143ae-f458-4251-aa44-f86bb534174e,Artisan Bread Loaf,Beauty,4.5885,Buzzshare,319,mockaroo,748deb5b11e228a55a00b77a4a61da72
cba03dd6-168f-4d77-b5d6-9087874dd445,Puzzle,Books,32.1885,Yabox,256,mockaroo,b34e6102228f941396989399e175045d
ba306da1-7036-4289-b823-268a75ab8df6,Electric Food Slicer,Baby,114.98849999999999,Blogspan,3,mockaroo,8f934488bff2682fa4c0bd518aa1bc60
96f58f07-5f38-4bbc-8d37-6651c8a6a7b0,Handmade Leather Journal,Automotive,45.9885,Realcube,481,mockaroo,49f4df3d2edf48f7dff8e86a817e7a98
468626f5-7df2-4f66-a180-c178b8681355,Portable Pet Water Bottle,Automotive,21.838499999999996,Myworks,390,mockaroo,0475ae535d7649dbd27869e5750a5776
fe24e923-9bed-470d-9e9b-bbe31726bbc0,Kids' Educational Tablet,Outdoors,149.4885,Brightbean,468,mockaroo,bb7b9ffdc075a138d956201922543a62
cc4df993-8e33-4912-8626-57fae4fc2e7f,Peach Mango Smoothie,Jewelry,4.0135,Youfeed,396,mockaroo,f557ec2547d87711d3e198e52b132963
f022c94b-fb92-42c4-b222-1c0d4fd6f222,Water Bottle with Built-in Fruit Infuser,Outdoors,21.838499999999996,Centidel,231,mockaroo,98ca8370e07de39096420d82f31ead33
4fe459ba-c44b-4dd9-962f-26dcc404bed3,Couscous,Toys,2.8635,Blogtags,76,mockaroo,466949f8c3cec7d549df0cf70a6fbc95
f2823689-0f6d-48be-8f08-82ea9442d3e4,Lemon Garlic Shrimp,Jewelry,10.3385,Blogtags,416,mockaroo,e426cc6c2888c1c8a0648cb17ef7e0a2


### Cell 9 — split into changed vs. brand new products

In [0]:
current_dim_product = spark.table("mia_catalog.gold.dim_product").filter(col("is_current") == True)

changed_products = (
    silver_changes.alias("src")
    .join(
        current_dim_product.select("product_key", col("record_hash").alias("existing_hash")).alias("dim"),
        on="product_key", how="inner"
    )
    .filter(col("record_hash") != col("existing_hash"))
    .select("src.*")
)

new_products = (
    silver_changes.alias("src")
    .join(current_dim_product.select("product_key").alias("dim"), on="product_key", how="left_anti")
)

print(f"Changed products: {changed_products.count()}")
print(f"Brand new products: {new_products.count()}")

Changed products: 15
Brand new products: 0


### Cell 10 — close out old versions of changed products

In [0]:
dim_product_table = DeltaTable.forName(spark, "mia_catalog.gold.dim_product")
changed_keys = [row["product_key"] for row in changed_products.select("product_key").collect()]

if changed_keys:
    changed_keys_sql = ", ".join([f"'{k}'" for k in changed_keys])
    dim_product_table.update(
        condition=f"product_key IN ({changed_keys_sql}) AND is_current = true",
        set={
            "is_current": "false",
            "effective_end_date": "current_date()",
            "dw_updated_at": "current_timestamp()"
        }
    )
    print(f"Closed out {len(changed_keys)} old versions")
else:
    print("No changed products this run")

Closed out 15 old versions


### Cell 11 — insert new versions (continuing surrogate keys correctly)

In [0]:
rows_to_insert = changed_products.unionByName(new_products)
insert_count = rows_to_insert.count()

if insert_count > 0:
    max_sk = spark.sql("SELECT COALESCE(MAX(product_sk), 0) as max_sk FROM mia_catalog.gold.dim_product").collect()[0]["max_sk"]
    window_spec = Window.orderBy("product_key")

    rows_final = (
        rows_to_insert
        .withColumn("product_sk", row_number().over(window_spec) + lit(max_sk))
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn("dw_created_at", current_timestamp())
        .withColumn("dw_updated_at", current_timestamp())
        .select("product_sk", "product_key", "product_name", "category", "price",
                 "brand", "stock_quantity", "source_system", "record_hash",
                 "effective_start_date", "effective_end_date", "is_current",
                 "dw_created_at", "dw_updated_at")
    )

    rows_final.write.format("delta").mode("append").saveAsTable("mia_catalog.gold.dim_product")
    print(f"Inserted {insert_count} new version rows, surrogate keys starting from {max_sk + 1}")
else:
    print("Nothing to insert this run")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Inserted 15 new version rows, surrogate keys starting from 1211


### Cell 12 — advance the watermark (only after successful write)

In [0]:
set_last_processed_version("mia_catalog.silver.silver_products", "mia_catalog.gold.dim_product", new_silver_version)
print(f"Watermark advanced to silver_products version {new_silver_version}")

Watermark advanced to silver_products version 11


### Cell 13 — verify: one product should now show 2 rows

In [0]:
display(spark.sql(f"""
    SELECT product_sk, product_key, product_name, price, is_current,
           effective_start_date, effective_end_date, dw_created_at
    FROM mia_catalog.gold.dim_product
    WHERE product_key = '{sample_product_keys[0]}'
    ORDER BY product_sk
"""))

display(spark.sql("""
    SELECT is_current, count(*) as row_count
    FROM mia_catalog.gold.dim_product
    GROUP BY is_current
"""))

product_sk,product_key,product_name,price,is_current,effective_start_date,effective_end_date,dw_created_at
1101,e82143ae-f458-4251-aa44-f86bb534174e,Artisan Bread Loaf,3.99,false,2026-08-15,2026-08-15,2026-08-15T09:48:09.756Z
1222,e82143ae-f458-4251-aa44-f86bb534174e,Artisan Bread Loaf,4.5885,true,2026-08-15,9999-12-31,2026-08-15T11:54:08.315Z


is_current,row_count
true,1195
false,30


In [0]:
%sql
select * from mia_catalog.silver.silver_products

product_key,product_name,category,price,brand,stock_quantity,created_at,rating,discount_percentage,dimensions_json,reviews_json,source_system,record_hash,last_updated_ts
4793db84-8d5a-4b05-886b-b4d801e51152,Dill Pickle Chips,Electronics,2.29,Riffpath,70,2026-07-15,null,null,null,null,mockaroo,65a55205daf9b223f239b2ffb1729d85,2026-08-12T08:00:19.218Z
f53586ab-36f6-4dba-b441-08ce69153379,Basil Tomato Soup,Sports,2.99,Browsebug,109,2025-12-01,null,null,null,null,mockaroo,0f9ba2c45900d62cdf27a41b892fcf23,2026-08-12T08:00:19.218Z
61fac6fb-2fad-4b25-8103-cf2b8362fad6,Fitbit Activity Tracker,Toys,99.99,Lazzy,147,2026-07-17,null,null,null,null,mockaroo,fcb6e48a7e72e44b5a3c2ae0c5ce4395,2026-08-12T08:00:19.218Z
eb7e9150-124f-4721-8ce8-ed02fd561be8,Adjustable Pedicure Footrest,Music,39.99,Yodo,316,2026-03-04,null,null,null,null,mockaroo,235748ef17b3f8551dc34adf3230f191,2026-08-12T08:00:19.218Z
5b17e9f3-a0c0-4083-847e-dcfe2aa45fbd,Frozen Berry Blend,Computers,4.99,Tazzy,169,2026-01-24,null,null,null,null,mockaroo,b53be950a9f14e1f0891055bf6c91c72,2026-08-12T08:00:19.218Z
93ed6529-9079-4777-a6ff-1f6d9b6347e2,Mini Meatballs,Automotive,5.99,Yombu,434,2025-10-27,null,null,null,null,mockaroo,1656f144b04240212df2a1ad7ef644a3,2026-08-12T08:00:19.218Z
78915807-529b-4ac0-8bcd-4b47af7c2517,Pineapple Chunks (canned),Books,2.29,Rooxo,4,2025-11-26,null,null,null,null,mockaroo,0e91727a573ff5885b517ec9f4c6220a,2026-08-12T08:00:19.218Z
c563ca80-dafe-4f5d-83ce-7250834316d9,Car Sunshade,Kids,19.99,Zoomdog,437,2025-12-07,null,null,null,null,mockaroo,554ce83e2c79fea225e9b71905235d48,2026-08-12T08:00:19.218Z
07348984-597c-41de-a1d9-90b281f2b0a8,Electric Griddle with Removable Plates,Games,59.99,Meeveo,436,2026-07-19,null,null,null,null,mockaroo,53d3df6df5385e9c8d28345f522532e7,2026-08-12T08:00:19.218Z
36c8d93e-a22e-470e-8d23-df6f81b7405c,Savory Quinoa Pudding,Automotive,2.99,Centidel,17,2026-07-30,null,null,null,null,mockaroo,1461bd5c0c027fe366074d30a23f2646,2026-08-12T08:00:19.218Z
